In [7]:
import pandas as pd

# 1. 데이터 로드
df_tags = pd.read_csv('steam_indie_tags.csv')
df_games = pd.read_csv('steam_indie_games.csv')

# 2. 지시사항에 따른 컬럼 제거
# - owners: steam_indie_games에 이미 존재하므로 제거
# - price: steam_indie_games와 데이터 불일치(1,866건) 및 기준 데이터 활용을 위해 제거
# - positive, negative: 이 역시 games 데이터에 존재하므로 중복 데이터 관리 차원에서 제거
drop_cols = ['owners', 'price', 'positive', 'negative', 'name']

# 존재하는 컬럼만 안전하게 제거
df_tags_cleaned = df_tags.drop(columns=[col for col in drop_cols if col in df_tags.columns])

# 3. 추가 전처리: 태그 데이터 파싱 준비 (선택 사항)
# tags 컬럼이 문자열 형태의 JSON/딕셔너리이므로 분석 시 바로 쓸 수 있게 정제하는 과정이 필요할 수 있습니다.

# 4. 결과 확인
print("전처리 완료 후 컬럼 목록:", df_tags_cleaned.columns.tolist())
print(f"최종 데이터 크기: {df_tags_cleaned.shape}")

# 5. 저장 (필요 시)
df_tags_cleaned.to_csv('steam_indie_tags_cleaned.csv', index=False)

전처리 완료 후 컬럼 목록: ['appid', 'developer', 'publisher', 'tags', 'updated_at']
최종 데이터 크기: (9706, 5)


In [8]:
import pandas as pd

# 1. 데이터 로드
df_games = pd.read_csv('steam_indie_games.csv')
df_tags = pd.read_csv('steam_indie_tags.csv')

# 2. Outer Join을 수행하여 양쪽의 모든 데이터를 합치고 출처를 표시 (_merge 컬럼 생성)
# suffixes를 통해 중복 컬럼(name 등)의 출처를 구분합니다.
merged_check = pd.merge(
    df_games[['appid', 'name']], 
    df_tags[['appid', 'name']], 
    on='appid', 
    how='outer', 
    suffixes=('_games', '_tags'),
    indicator=True
)

# 3. 미스매칭 데이터 추출 (양쪽에 모두 존재하지 않는 경우)
mismatch = merged_check[merged_check['_merge'] != 'both']

# 4. 결과 분류
# (1) Tags에는 있는데 Games에는 없는 경우 (right_only)
right_only = mismatch[mismatch['_merge'] == 'right_only']

# (2) Games에는 있는데 Tags에는 없는 경우 (left_only)
left_only = mismatch[mismatch['_merge'] == 'left_only']

# 5. 리포트 출력
print("="*60)
print(f"🔎 미스매칭 분석 결과 (전체 미스매칭: {len(mismatch)}건)")
print("="*60)
print(f"1. [Tags 전용] Tags 파일에만 있고 Games에는 없는 게임: {len(right_only)}건")
if not right_only.empty:
    print(right_only[['appid', 'name_tags']].head(10)) # 상위 10개 출력
    
print("-" * 60)
print(f"2. [Games 전용] Games 파일에만 있고 Tags에는 없는 게임: {len(left_only)}건")
if not left_only.empty:
    print(left_only[['appid', 'name_games']].head(10))
else:
    print("-> 모든 Games 데이터가 Tags 파일에 존재합니다.")
print("="*60)

df_tags_cleaned['updated_at'].dtype

# 필요하다면 미스매칭 리스트를 엑셀이나 CSV로 저장해서 팀원과 공유하세요.
# mismatch.to_csv('mismatch_check.csv', index=False)

🔎 미스매칭 분석 결과 (전체 미스매칭: 14건)
1. [Tags 전용] Tags 파일에만 있고 Games에는 없는 게임: 14건
        appid                              name_tags
323   1049590                         Eternal Return
395   1116540                      DAVIGO: VR vs. PC
527   1213300           Rebellion GODSOUL: Awakening
1317  1623730                               Palworld
1491  1700300              World Warfare & Economics
2156  1948800     Yi Xian: The Cultivation Card Game
2224  1966720                         Lethal Company
2708  2115850                           Last Holiday
2921  2162800                     shapez 2 - Factory
3037  2186320  Ages of Conflict: World War Simulator
------------------------------------------------------------
2. [Games 전용] Games 파일에만 있고 Tags에는 없는 게임: 0건
-> 모든 Games 데이터가 Tags 파일에 존재합니다.


<StringDtype(storage='python', na_value=nan)>

In [9]:
df_tags_cleaned.head()

,appid,developer,publisher,tags,updated_at
0,1432860,Pixel Sprout Studios,Pixel Sprout Studios,"{""RPG"": 442, ""Magic"": 324, ""Combat"": 287, ""Min...",2026-04-23 13:58:47.087
1,1473350,Myco,(Myco),"{""2D"": 117, ""Cute"": 133, ""Idler"": 193, ""Indie""...",2026-04-23 13:58:48.495
2,1993150,烟水寒工作室,烟水寒工作室,"{""3D"": 359, ""RPG"": 409, ""Indie"": 332, ""Space"":...",2026-04-23 13:58:49.904
3,2527500,AIHASTO,"IndieArk, Shochiku (Japan)","{""2D"": 761, ""3D"": 1392, ""RPG"": 815, ""Cute"": 21...",2026-04-23 13:58:51.311
4,1169040,Fair Games ApS,Fair Games ApS,"{""2D"": 188, ""RPG"": 241, ""Co-op"": 237, ""Indie"":...",2026-04-23 13:58:52.692


In [11]:
df_tags_cleaned['updated_at'].info()

<class 'pandas.Series'>
RangeIndex: 9706 entries, 0 to 9705
Series name: updated_at
Non-Null Count  Dtype
--------------  -----
9706 non-null   str  
dtypes: str(1)
memory usage: 76.0 KB


In [13]:
import pandas as pd

# 1. 데이터 로드
df_tags = pd.read_csv('steam_indie_tags.csv')
df_games = pd.read_csv('steam_indie_games.csv')

# 2. 제거 대상 컬럼 확정
# - owners, price, positive, negative, name: games 데이터와 중복/불일치
# - updated_at: 분석 불필요 판단으로 제거
drop_cols = ['owners', 'price', 'positive', 'negative', 'name', 'updated_at']

# 3. 컬럼 제거 실행
df_tags_cleaned = df_tags.drop(columns=[col for col in drop_cols if col in df_tags.columns])

# 4. 결과 확인
print("--- [steam_indie_tags] 전처리 완료 ---")
print(f"남은 컬럼: {df_tags_cleaned.columns.tolist()}")
# 예상 결과: ['appid', 'developer', 'publisher', 'tags']

# 5. [옵션] steam_indie_games와 바로 병합하기
# 이렇게 하면 나중에 다른 분석할 때 매우 편합니다.
#df_integrated = pd.merge(df_games, df_tags_cleaned, on='appid', how='inner')

print(f"\n최종 통합 데이터 행 수: {len(df_tags_cleaned):,}개")
print(f"최종 통합 데이터 컬럼: {df_tags_cleaned.columns.tolist()}")

# 6. 저장
df_tags_cleaned.to_csv('steam_indie_tags_clean.csv', index=False)

--- [steam_indie_tags] 전처리 완료 ---
남은 컬럼: ['appid', 'developer', 'publisher', 'tags']

최종 통합 데이터 행 수: 9,706개
최종 통합 데이터 컬럼: ['appid', 'developer', 'publisher', 'tags']


In [ ]:
df_tags_cleaned.head()
df_tags_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 9706 entries, 0 to 9705
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   appid      9706 non-null   int64
 1   developer  9694 non-null   str  
 2   publisher  9673 non-null   str  
 3   tags       9706 non-null   str  
dtypes: int64(1), str(3)
memory usage: 303.4 KB


: 